# Context selector on the AACR 50% set: small and large LLMs (Colab A100)

The same question, data, label, folds, policies and scoring as `selector50_nonllm.ipynb`; the
selectors here read the hunk with a code LLM. Small and large models of one family
(Qwen2.5-Coder) read the same text of the same length (2,048 tokens), so differences between
them come from model size alone.

| Family | Models | How it scores a hunk |
|---|---|---|
| zero-shot | 1.5B, 7B, 14B | log P("Yes") - log P("No") to "would more code help the reviewer?"; no training |
| embedding | 0.5B, 1.5B, 3B, 7B | mean-pooled last hidden state -> PCA(32) in each fold -> logistic regression; also with the tabular features |
| LoRA | 0.5B, 3B (7B optional) | one-output head, LoRA r = 16, class-weighted loss, 3 epochs, 3 seeds averaged by rank |
| baseline | tabular, logistic | the pre-declared baseline, recomputed here |

**Runtime on an A100 (estimates):** zero-shot about 2 min (1.5B), 6 min (7B), 12 min (14B);
embeddings 1 to 6 min each; LoRA about 30 min (0.5B) and 1.5 to 2 h (3B) for 3 seeds x 5 folds.
Every model's output is cached on Drive per model, seed and fold, so a lost session resumes
where it stopped: just Run all again.

**Before running:** Runtime -> Change runtime type -> A100 GPU. Put the folder
`aacr_hunk_50%_selector_files` on Drive (path in the first cell). To put the non-LLM models in
the same tables, copy `results/selector50_nonllm/nonllm_oof_scores.csv` from the laptop run into
the results folder on Drive (`selector50_llm_results`); the folds are checked to be identical.

Quick local check: `SELECTOR_SMOKE=1` (0.5B only, short texts; the numbers mean nothing).

## 1. Setup

In [ ]:
import os, sys, json, hashlib, zipfile, time, math, warnings, collections, gc
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB and not any(k in sys.prefix for k in ("/envs/cs536", "/.venv")):
    raise RuntimeError(
        f"Wrong kernel: {sys.executable}\n"
        "In VS Code: kernel picker (top right) -> Select Another Kernel -> Python Environments -> "
        "cs536 (~/anaconda3/envs/cs536/bin/python), then Restart and Run All.")
print("Python:", sys.executable)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

def repo_root():
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "AGENTS.md").exists():
            return d
    return Path.cwd()
REPO = None if IN_COLAB else repo_root()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Colab's preinstalled peft raises "incompatible version of torchao" when it adds LoRA;
    # peft >= 0.22 only warns. Nothing here uses torchao, so remove it too.
    os.system(f'"{sys.executable}" -m pip -q install -U "peft>=0.22" "accelerate>=0.34"')
    os.system(f'"{sys.executable}" -m pip -q uninstall -y torchao')

# Folder (or zip) with hunk_outcomes.json, coverage.csv and context_levels_slim.jsonl.
COLAB_DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/LH/ML2/aacr_hunk_50%_selector_files"
LOCAL_DATA_PATH = str(REPO / "data" / "aacr_hunk_50%_selector_files") if REPO else ""
DATA_PATH = os.environ.get("SELECTOR_DATA", COLAB_DATA_PATH if IN_COLAB else LOCAL_DATA_PATH)
# Results and the model caches go here (on Drive, so a lost session resumes where it stopped).
OUT_DIR = Path(os.environ.get("SELECTOR_OUT", "/content/drive/MyDrive/Colab Notebooks/LH/ML2/selector50_llm_results"
                              if IN_COLAB else str(REPO / "results" / "selector50_llm")))
# Optional: the non-LLM notebook's out-of-fold scores, to put its models in this comparison.
# Copy results/selector50_nonllm/nonllm_oof_scores.csv here (same folder as the results).
NONLLM_OOF = OUT_DIR / "nonllm_oof_scores.csv"

if IN_COLAB and not Path(DATA_PATH).exists():
    from google.colab import files as colab_files
    print(f"{DATA_PATH} not found. Choose hunk_outcomes.json, context_levels_slim.jsonl and coverage.csv:")
    DATA_PATH = "/content/selector50_data"
    Path(DATA_PATH).mkdir(exist_ok=True)
    for name, data in colab_files.upload().items():
        (Path(DATA_PATH) / name).write_bytes(data)

def find_inputs(path):
    """The three input files, found by name under a folder (a zip is unzipped first)."""
    p = Path(path)
    if p.is_file() and p.suffix == ".zip":
        dest = Path("/content/selector50_data" if IN_COLAB else p.with_suffix(""))
        if not dest.exists():
            print(f"Unzipping {p} -> {dest}")
            with zipfile.ZipFile(p) as z:
                z.extractall(dest)
        p = dest
    if not p.is_dir():
        raise FileNotFoundError(f"DATA_PATH does not exist: {p}")
    def first(*names):
        for name in names:
            hits = sorted(h for h in p.rglob(name) if "__MACOSX" not in h.parts and not h.name.startswith("._"))
            if hits:
                return hits[0]
        return None
    files = {"outcomes": first("hunk_outcomes.json"),
             "contexts": first("context_levels_slim.jsonl", "context_levels.jsonl"),
             "coverage": first("coverage.csv")}
    missing = [k for k, v in files.items() if v is None]
    if missing:
        raise FileNotFoundError(f"Missing {missing} under {p}: need hunk_outcomes.json, "
                                "context_levels_slim.jsonl and coverage.csv")
    return files

FILES = find_inputs(DATA_PATH)
for k, v in FILES.items():
    print(f"{k:9s} {v}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results go to", OUT_DIR)

CACHE_DIR = OUT_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
transformers.logging.set_verbosity_error()

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__,
      "| torch", torch.__version__, "| transformers", transformers.__version__)
if DEVICE == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"GPU: {GPU_NAME} ({GPU_GB:.0f} GB), bfloat16: {torch.cuda.is_bf16_supported()}")
    if GPU_GB < 35:
        print("WARNING: sized for an A100 (40 or 80 GB). Drop the 14B and 7B models from the lists below.")
else:
    GPU_NAME = DEVICE
    print(f"No CUDA GPU ({DEVICE}). Use SMOKE = True below for a quick check; the full run needs an A100.")

## 2. Configuration

Change settings here, not further down. Drop models from the lists to save time.

In [ ]:
LEVELS = ["H01_min", "H02_function", "H03_file", "H04_callees1"]   # the four reviewed levels
BASE_LEVEL = "H01_min"                                           # every hunk's default
UPGRADE_LEVELS = ["H02_function", "H03_file", "H04_callees1"]    # a selected hunk goes to one of these
MAIN_UP = "H02_function"     # the upgrade level of the main comparison (cheapest way to the gain, aacr_50pct_run.md)

LABEL = "wider"              # "wider": some wider level covers more references than H01_min (190 hunks);
                             # "wider_weighted": the same with severity weights
WEIGHTINGS = {"flat": (1, 1, 1), "linear": (3, 2, 1), "agreed": (5, 2, 0.5), "steep": (25, 5, 1),
              "blocking_only": (1, 0, 0)}
TARGET_WEIGHTING = "agreed"  # the severity score used for every budget result

# Folds: CV = "group5" (5 folds of whole repositories, same rule in both notebooks) or "repo"
# (leave one repository out, 37 folds). Keep "group5" to compare with the LLM notebook.
CV = "group5"
N_FOLDS = 5
FOLD_SEED = 0

# Budgets: every hunk starts at BASE_LEVEL; a selector spends a share of the extra tokens that
# upgrading every hunk would cost. 1.0 is always the upgrade level.
SPEND = [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]
TARGET_SPEND = 0.3           # 20% of hunks are positive; 30% of the extra tokens is the main point
N_RANDOM = 1000
N_BOOT = 2000
REFERENCE_METHOD = "tabular, logistic"     # the pre-declared baseline every method is compared with

# Models. Small and large models of one family (Qwen2.5-Coder, same tokenizer), all reading the
# same text of the same length, so the comparison is between model sizes only.
ZS_MODELS = [                       # zero-shot: log P(Yes) - log P(No); no training
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Qwen/Qwen2.5-Coder-14B-Instruct",      # about 30 GB in bfloat16; drop on a 40 GB GPU if it runs out of memory
]
EMB_MODELS = [                      # frozen mean-pooled embeddings -> PCA -> logistic regression
    "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    "Qwen/Qwen2.5-Coder-7B-Instruct",
]
FT_MODELS = [                       # LoRA fine-tuning with a one-output head, 3 seeds x 5 folds
    "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    # "Qwen/Qwen2.5-Coder-7B-Instruct",     # about 4 more hours on an A100; uncomment if time allows
]
MAX_LEN = 2048                # tokens of hunk text, the same for every model
DIFF_SHARE = 0.6              # at least this share of MAX_LEN goes to the diff when the text is too long
USE_META = True               # a first line of input-side facts (path, language, sizes, context tokens)

RUN_ZERO_SHOT, RUN_EMBED, RUN_FINETUNE = True, True, True

ZS_BATCH = 4
EMB_BATCH = 8
EMB_PCA = 32                  # PCA dimensions, fitted inside each fold

FT_SEEDS = [0, 1, 2]
FT_EPOCHS = 3
FT_LR = 2e-4
FT_BATCH = 4
FT_GRAD_ACC = 4               # effective batch 16
FT_WARMUP = 0.1
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
GRAD_CKPT = True              # gradient checkpointing: less memory, about 30% slower

# SMOKE = True: a quick check of the whole notebook on a laptop or a small GPU (0.5B only,
# short texts, LoRA on a few hunks). Its numbers mean nothing. Caches go to a separate folder.
SMOKE = os.environ.get("SELECTOR_SMOKE", "0") == "1"
if SMOKE:
    ZS_MODELS = EMB_MODELS = FT_MODELS = ["Qwen/Qwen2.5-Coder-0.5B-Instruct"]
    MAX_LEN, FT_SEEDS, FT_EPOCHS, N_RANDOM, N_BOOT = 256, [0], 1, 100, 200
    CACHE_DIR = OUT_DIR / "cache_smoke"; CACHE_DIR.mkdir(exist_ok=True)
FT_SMOKE_TRAIN = 48           # training hunks per fold in SMOKE mode
if DEVICE == "mps":           # Apple GPU: float32 training needs a small batch
    FT_BATCH, FT_GRAD_ACC = 1, 16

# Fit the model lists to the GPU Colab gave you, so the notebook runs without edits on any of them.
def _size(name):
    return float(name.split("-Coder-")[1].split("B")[0])
if DEVICE == "cuda" and not SMOKE:
    if GPU_GB >= 70:          # A100 80 GB: everything as listed
        pass
    elif GPU_GB >= 35:        # A100 40 GB: all models, smaller batches
        ZS_BATCH, EMB_BATCH, FT_BATCH, FT_GRAD_ACC = 2, 4, 2, 8
    elif GPU_GB >= 20:        # L4 24 GB (or A10): up to 7B for inference, up to 3B for LoRA
        ZS_MODELS = [m for m in ZS_MODELS if _size(m) <= 7]
        EMB_MODELS = [m for m in EMB_MODELS if _size(m) <= 7]
        FT_MODELS = [m for m in FT_MODELS if _size(m) <= 3]
        ZS_BATCH, EMB_BATCH, FT_BATCH, FT_GRAD_ACC = 1, 2, 1, 16
    else:                     # T4 16 GB: up to 3B for inference, 0.5B for LoRA
        ZS_MODELS = [m for m in ZS_MODELS if _size(m) <= 3]
        EMB_MODELS = [m for m in EMB_MODELS if _size(m) <= 3]
        FT_MODELS = [m for m in FT_MODELS if _size(m) <= 1]
        ZS_BATCH, EMB_BATCH, FT_BATCH, FT_GRAD_ACC = 1, 2, 1, 16
    print(f"GPU {GPU_NAME} ({GPU_GB:.0f} GB) -> zero-shot {[_size(m) for m in ZS_MODELS]}B, "
          f"embeddings {[_size(m) for m in EMB_MODELS]}B, LoRA {[_size(m) for m in FT_MODELS]}B; "
          f"batches zs {ZS_BATCH}, emb {EMB_BATCH}, LoRA {FT_BATCH} x {FT_GRAD_ACC}")

FT_TAG = hashlib.sha1(json.dumps([FT_EPOCHS, FT_LR, FT_BATCH * FT_GRAD_ACC, FT_WARMUP, LORA_R, LORA_ALPHA,
                                  LORA_DROPOUT, LABEL, SMOKE]).encode()).hexdigest()[:8]
PLOT_METHODS = ["tabular, logistic", "zero-shot 1.5B", "zero-shot 7B", "zero-shot 14B",
                "embedding 0.5B", "embedding 7B", "LoRA 0.5B", "LoRA 3B"]
FIG_NAME = "smoke_frontier_llm.png" if SMOKE else "frontier_llm.png"

## 3. Load the outcome table and check it against `coverage.csv`

In [ ]:
with open(FILES["outcomes"]) as f:
    OUT = json.load(f)
assert OUT["levels"] == LEVELS, OUT["levels"]
rows = OUT["rows"]
hunks = pd.DataFrame([{k: v for k, v in r.items() if k != "levels"} for r in rows])
hunks["repo"] = hunks["pr"].str.rsplit("__", n=1).str[0]
H = len(hunks)
hunk_pos = {h: i for i, h in enumerate(hunks["hunk"])}

ref_rows = [{"pr": pr, "ref": int(k), **r} for pr, refs in OUT["refs"].items() for k, r in refs.items()]
refs_df = pd.DataFrame(ref_rows).sort_values(["pr", "ref"]).reset_index(drop=True)
refs_df["repo"] = refs_df["pr"].str.rsplit("__", n=1).str[0]
R = len(refs_df)
ref_pos = {(p, k): i for i, (p, k) in enumerate(zip(refs_df["pr"], refs_df["ref"]))}

L = len(LEVELS)
COV = np.zeros((H, L, R), dtype=bool)     # COV[h, l, r]: hunk h's review at level l covers reference r
TOK = np.zeros((H, L))                    # reviewer input tokens of hunk h at level l
for r in rows:
    i = hunk_pos[r["hunk"]]
    for j, lv in enumerate(LEVELS):
        c = r["levels"][lv]
        TOK[i, j] = c["tokens"]
        for k in c["covered"]:
            COV[i, j, ref_pos[(r["pr"], k)]] = True

# Check: every (PR, level) union equals coverage.csv.
cc = pd.read_csv(FILES["coverage"])
cc = cc[cc["covered"].astype(str) == "True"]
want = collections.defaultdict(set)
for e, lv, k in zip(cc["entry"], cc["level"], cc["reference"]):
    want[(e, lv)].add(int(k))
got = collections.defaultdict(set)
for r in rows:
    for lv in LEVELS:
        got[(r["pr"], lv)].update(r["levels"][lv]["covered"])
bad = [k for k in set(want) | set(got) if want[k] != got[k]]
assert not bad, f"{len(bad)} (PR, level) unions differ from coverage.csv, e.g. {bad[:3]}"

# Severity weights. Two references have no severity label: weight 0 in every weighted score,
# as in the run's own severity score (docs/aacr_50pct_run.md, Section 1).
TIER_IDX = refs_df["tier"].map({"blocking": 0, "should_fix": 1, "nit": 2})
W = {name: np.array([0.0 if pd.isna(t) else w[int(t)] for t in TIER_IDX]) for name, w in WEIGHTINGS.items()}
W["flat"] = np.ones(R)
IS_HUMAN = refs_df["human"].astype(bool).to_numpy()
print(f"{hunks['pr'].nunique()} PRs, {hunks['repo'].nunique()} repositories, {H} hunks, {R} references "
      f"({IS_HUMAN.sum()} written by a person); unlabelled severity: {int(TIER_IDX.isna().sum())}")
print("coverage.csv check: every (PR, level) union matches.")

## 4. Input-side features and the LLM's text

The LLM reads one line of input-side facts, then the diff, then the PR text (the `H01_min` text), cut to `MAX_LEN` tokens. Nothing from the references, the reviews or the judge enters a feature or the text.

In [ ]:
CODE_EXT = {".c", ".h", ".cc", ".cpp", ".hpp", ".cxx", ".cs", ".go", ".java", ".kt", ".js",
            ".jsx", ".ts", ".tsx", ".mjs", ".py", ".rs", ".php", ".rb", ".swift", ".scala"}

hunk_feat, pr_feat, H01_TEXT = {}, {}, {}
ALL_LEVELS = None
with open(FILES["contexts"]) as f:
    for line in f:
        pr = json.loads(line)
        if pr.get("status") != "ok":
            continue
        s = pr["stats"]
        pr_feat[pr["id"]] = {
            "pr_changed_files": s.get("changed_files", 0), "pr_diff_chars": s.get("diff_chars", 0),
            "pr_title_chars": s.get("pr_title_chars", 0), "pr_body_chars": s.get("pr_body_chars", 0),
            "pr_guide_files": s.get("guide_files", 0), "pr_search_hits": s.get("search_hits", 0),
            "pr_changed_symbols": s.get("changed_symbols", 0),
            "pr_graph_nodes": pr.get("graph", {}).get("nodes", s.get("graph_nodes", 0)),
            "pr_graph_edges": pr.get("graph", {}).get("edges", s.get("graph_edges", 0))}
        for h in pr["hunks"]:
            if h["id"] not in hunk_pos:
                continue
            hs = h["stats"]
            ALL_LEVELS = ALL_LEVELS or list(h["levels"])
            d = {"hunk_index": h["index"], "old_count": h.get("old_count", 0), "new_count": h.get("new_count", 0),
                 "search_names": hs.get("search_names", 0)}
            for key in ("callees_by_hop", "callers_by_hop", "test_callers_by_hop"):
                for hop in ("1", "2", "3"):
                    d[f"{key}_{hop}"] = (hs.get(key) or {}).get(hop, 0)
            for kind in ("Function", "Method", "Class", "Interface", "Variable", "Module", "Type", "Enum", "Macro"):
                d[f"sym_{kind}"] = (hs.get("changed_symbols_by_label") or {}).get(kind, 0)
            # Every built level's size is known before any review (only four were reviewed).
            for lv in ALL_LEVELS:
                x = h["levels"][lv]
                d[f"tok_{lv}"] = x["input_tokens"]
                d[f"ctx_{lv}"] = x["context_tokens"]
                d[f"items_{lv}"] = x["items"]
                d[f"same_{lv}"] = int(bool(x["same_as_parent"]))
                d[f"capped_{lv}"] = int(bool(x["capped"]))
            for kind, n in (h["levels"][ALL_LEVELS[-1]].get("included_by_kind") or {}).items():
                d[f"kindall_{kind}"] = n
            hunk_feat[h["id"]] = d
            H01_TEXT[h["id"]] = h["levels"]["H01_min"]["text"]

missing = set(hunks["hunk"]) - set(hunk_feat)
assert not missing, f"{len(missing)} hunks are not in the contexts file"
assert all(H01_TEXT[h] for h in hunks["hunk"]), "some H01_min texts are empty"

F = pd.DataFrame([hunk_feat[h] for h in hunks["hunk"]]).fillna(0)
P = pd.DataFrame([pr_feat[p] for p in hunks["pr"]])
base_cols = ["pr_hunks", "file_hunks", "added", "removed", "diff_chars", "changed_symbols",
             "callees1", "callers1", "tests1", "search_hits", "new_start"]
X_tab = pd.concat([hunks[base_cols].reset_index(drop=True).astype(float), F, P], axis=1)
X_tab["new_file"] = hunks["new_file"].astype(int).to_numpy()
X_tab["deleted_file"] = hunks["deleted_file"].astype(int).to_numpy()
path = hunks["file"].fillna("")
X_tab["path_depth"] = path.str.count("/").to_numpy()
IS_TEST_PATH = path.str.contains(r"(?i)(?:^|/)(?:tests?|spec|__tests__)(?:/|$)|_test\.|\.test\.|test_|\.spec\.").to_numpy()
X_tab["path_is_test"] = IS_TEST_PATH.astype(int)
X_tab["is_code"] = hunks["ext"].isin(CODE_EXT).astype(int).to_numpy()
X_tab["hunk_rel_pos"] = X_tab["hunk_index"] / X_tab["pr_hunks"].clip(lower=1)
for lv in ALL_LEVELS[1:]:
    X_tab[f"ratio_{lv}"] = X_tab[f"tok_{lv}"] / X_tab["tok_H01_min"].clip(lower=1)
for c in list(X_tab.columns):
    if c.startswith(("tok_", "ctx_", "pr_", "callees", "callers", "test_callers", "search", "diff_chars",
                     "kindall_", "items_", "new_start", "added", "removed", "old_count", "new_count")):
        X_tab[c] = np.log1p(X_tab[c].astype(float).clip(lower=0))
LANG = pd.get_dummies(hunks["language"], prefix="lang").astype(float).reset_index(drop=True)
X_tab = pd.concat([X_tab, LANG], axis=1).astype(float)
X_tab = X_tab.loc[:, X_tab.std() > 0]

FORBIDDEN = ("category", "context_label", "human", "tier", "covered", "finding", "matched",
             "audit", "reference", "recall", "precision", "score", "judge", "review", "out_tok")
leaks = [c for c in X_tab.columns if any(w in c.lower() for w in FORBIDDEN)]
assert not leaks, f"answer-side columns in the features: {leaks}"
XT = X_tab.to_numpy()
XL = LANG.to_numpy()
TEXT_DOC = [f"{p}\n{H01_TEXT[h]}" for p, h in zip(path, hunks["hunk"])]   # path + H01_min text
print(f"Tabular features: {X_tab.shape[1]} (from {len(ALL_LEVELS)} built levels)")

In [ ]:
def meta_line(i):
    h, f = hunks.iloc[i], F.iloc[i]
    extra = {lv: int(f[f"ctx_{lv}"] - f["ctx_H01_min"]) for lv in UPGRADE_LEVELS}
    return (f"File: {h['file']} | language: {h['language']} | "
            f"{'new file | ' if h['new_file'] else ''}{'test file | ' if IS_TEST_PATH[i] else ''}"
            f"+{int(h['added'])} -{int(h['removed'])} lines | hunk {int(f['hunk_index'])} of {int(h['pr_hunks'])} in the PR, "
            f"{int(h['file_hunks'])} in this file | callers {int(h['callers1'])}, callees {int(h['callees1'])}, "
            f"tests {int(h['tests1'])} | context tokens: function {extra['H02_function']}, "
            f"file {extra['H03_file']}, callees {extra['H04_callees1']}")
META = [meta_line(i) for i in range(H)]
TEXTS = [H01_TEXT[h] for h in hunks["hunk"]]
print("Example meta line:\n ", META[0])

def split_h01(t):
    """(PR part, diff part) of an H01_min text."""
    k = t.rfind("## Diff")
    return (t[:k].strip(), t[k:].strip()) if k >= 0 else (t.strip(), "")

_TEXT_CACHE = {}
def build_texts(tok, max_len):
    """Meta line, then the diff, then the PR text, cut to max_len tokens; the diff keeps at least
    DIFF_SHARE of the room when both are long."""
    key = (tok.name_or_path, max_len)
    if key in _TEXT_CACHE:
        return _TEXT_CACHE[key]
    out = []
    for i in range(H):
        pr_part, diff = split_h01(TEXTS[i])
        head = (META[i] + "\n\n") if USE_META else ""
        n_head = len(tok(head, add_special_tokens=False).input_ids)
        d_ids = tok(diff, add_special_tokens=False).input_ids
        p_ids = tok(pr_part, add_special_tokens=False).input_ids
        room = max(16, max_len - n_head - 8)
        if len(d_ids) + len(p_ids) > room:
            keep_d = min(len(d_ids), max(int(room * DIFF_SHARE), room - len(p_ids)))
            d_ids, p_ids = d_ids[:keep_d], p_ids[:room - keep_d]
        out.append(head + tok.decode(d_ids) + "\n\n" + tok.decode(p_ids))
    _TEXT_CACHE[key] = out
    return out

## 5. Label and folds

The fold hash must equal the non-LLM notebook's.

In [ ]:
LIDX = {lv: j for j, lv in enumerate(LEVELS)}
NCOV = COV.sum(axis=2)                                         # (H, L) references covered
GAIN = (COV * W[TARGET_WEIGHTING][None, None, :]).sum(axis=2)  # (H, L) severity-weighted
# Label (docs/aacr_50pct_run.md, Section 3): some wider level covers more references than H01_min.
Y_COUNT = (NCOV[:, 1:].max(axis=1) > NCOV[:, 0]).astype(int)
Y_WEIGHTED = (GAIN[:, 1:].max(axis=1) > GAIN[:, 0]).astype(int)
Y = Y_COUNT if LABEL == "wider" else Y_WEIGHTED
assert LABEL != "wider" or Y.sum() == 190, Y.sum()
print(f"Label '{LABEL}': {Y.sum()} of {H} hunks positive "
      f"(count-based {Y_COUNT.sum()}, severity-weighted {Y_WEIGHTED.sum()})")

# Folds: repositories split by a fixed rule in plain Python, so every machine and every
# scikit-learn version gets the same folds (the earlier runs did not; selector_llm_a100_run.md).
# Repositories are taken largest first (ties by a seeded hash) and each goes to the fold with
# the fewest hunks so far.
REPOS = hunks["repo"].to_numpy()
def make_folds(repos, k, seed=0):
    counts = collections.Counter(repos)
    key = lambda r: hashlib.sha1(f"{seed}:{r}".encode()).hexdigest()
    load, fold_of = [0] * k, {}
    for r in sorted(counts, key=lambda r: (-counts[r], key(r))):
        j = min(range(k), key=lambda j: (load[j], j))
        fold_of[r] = j
        load[j] += counts[r]
    return fold_of
if CV == "repo":
    FOLD_OF = {r: j for j, r in enumerate(sorted(set(REPOS)))}
else:
    FOLD_OF = make_folds(REPOS, N_FOLDS, FOLD_SEED)
FOLD_ID = np.array([FOLD_OF[r] for r in REPOS])
FOLDS = [(np.flatnonzero(FOLD_ID != j), np.flatnonzero(FOLD_ID == j)) for j in sorted(set(FOLD_ID))]
for tr, te in FOLDS:
    assert not set(REPOS[tr]) & set(REPOS[te])
FOLD_HASH = hashlib.sha1(",".join(map(str, FOLD_ID)).encode()).hexdigest()[:10]
print(f"{len(FOLDS)} folds ({CV}), fold hash {FOLD_HASH}; hunks per test fold:",
      [len(te) for _, te in FOLDS], "positives:", [int(Y[te].sum()) for _, te in FOLDS])

## 6. Policies, fixed levels and baselines

In [ ]:
b_i = LIDX[BASE_LEVEL]

def evaluate(choice):
    """Quality and cost of reviewing hunk i at level choice[i]; a PR's covered references are
    the union over its hunks."""
    covered = COV[np.arange(H), choice, :].any(axis=0)
    out = {"covered": int(covered.sum()), "recall": covered.mean(),
           "tokens": TOK[np.arange(H), choice].sum(),
           "recall_human": covered[IS_HUMAN].mean(), "recall_ai": covered[~IS_HUMAN].mean()}
    for name, w in W.items():
        out[f"score_{name}"] = (covered * w).sum() / w.sum()
    return out

def fixed(level):
    return np.full(H, LIDX[level])

FIXED = pd.DataFrame([{"policy": f"always {lv}", **evaluate(fixed(lv))} for lv in LEVELS])
assert FIXED["covered"].tolist() == [342, 363, 357, 356], FIXED["covered"].tolist()
print("Fixed levels match docs/aacr_50pct_run.md, Section 2.")
FIXED["token_share_vs_H01"] = FIXED["tokens"] / FIXED.loc[b_i, "tokens"]

def upgrade_choice(order, up, f):
    """Start every hunk at BASE_LEVEL; upgrade hunks to level `up` in this order until a share f
    of the extra tokens of always `up` is spent."""
    u = LIDX[up]
    extra = TOK[:, u] - TOK[:, b_i]
    budget = f * extra.clip(min=0).sum()
    k = int(np.searchsorted(np.cumsum(extra[order]), budget + 1e-9, side="right"))
    ch = fixed(BASE_LEVEL).copy(); ch[order[:k]] = u
    return ch

def order_of(score):
    return np.argsort(-np.asarray(score, dtype=float), kind="stable")

def frontier(score, up):
    o = order_of(score)
    rows_ = []
    for f in SPEND:
        e = evaluate(upgrade_choice(o, up, f))
        e["token_share"] = e["tokens"] / FIXED.loc[LIDX[up], "tokens"]
        rows_.append({"upgrade_to": up, "spend": f, **e})
    return pd.DataFrame(rows_)

# Baselines that need no learning, per upgrade level.
CHEAP = {up: -(TOK[:, LIDX[up]] - TOK[:, b_i]) for up in UPGRADE_LEVELS}                    # cheapest first
ORACLE = {up: (GAIN[:, LIDX[up]] - GAIN[:, b_i]) - 1e-6 * (TOK[:, LIDX[up]] - TOK[:, b_i])  # hindsight
          for up in UPGRADE_LEVELS}
RAND = {}
for up in UPGRADE_LEVELS:
    for f in SPEND:
        v = pd.DataFrame([evaluate(upgrade_choice(np.random.default_rng(k).permutation(H), up, f))
                          for k in range(N_RANDOM)])
        RAND[(up, f)] = {"covered_mean": v["covered"].mean(), "covered_p95": v["covered"].quantile(.95),
                         "score_mean": v[f"score_{TARGET_WEIGHTING}"].mean(),
                         "score_p95": v[f"score_{TARGET_WEIGHTING}"].quantile(.95)}

# Hindsight per hunk over all four levels: the cheapest level that covers the most references.
best = np.array([min(np.flatnonzero(NCOV[i] == NCOV[i].max()), key=lambda j: TOK[i, j]) for i in range(H)])
e = evaluate(best)
print(f"Hindsight best level per hunk: {e['covered']} of {R} references at "
      f"{e['tokens'] / FIXED.loc[b_i, 'tokens']:.2f}x the tokens of {BASE_LEVEL} "
      f"(docs/aacr_50pct_run.md, Section 3: 459 at 1.5x)")
print(FIXED[["policy", "covered", f"score_{TARGET_WEIGHTING}", "recall_human", "recall_ai", "token_share_vs_H01"]]
      .round(3).to_string(index=False))

In [ ]:
def cluster_boot_auc(y, s, groups, n=None, seed=0):
    """95% interval of the AUC, resampling repositories."""
    n = n or N_BOOT
    g = np.unique(groups); rng = np.random.default_rng(seed); vals = []
    by = {k: np.flatnonzero(groups == k) for k in g}
    for _ in range(n):
        idx = np.concatenate([by[k] for k in rng.choice(g, size=len(g))])
        if y[idx].min() != y[idx].max():
            vals.append(roc_auc_score(y[idx], s[idx]))
    return np.percentile(vals, [2.5, 97.5])

def boot_diff(ch_a, ch_b, n=None, seed=0):
    """Paired difference in weighted score and in covered references, resampling repositories."""
    n = n or N_BOOT
    g = np.unique(REPOS); rng = np.random.default_rng(seed); w = W[TARGET_WEIGHTING]
    rmask = {r: (refs_df["repo"] == r).to_numpy() for r in g}
    def cov(ch, r):
        idx = np.flatnonzero(REPOS == r)
        return COV[idx, ch[idx], :].any(0) & rmask[r]
    ca = {r: cov(ch_a, r) for r in g}; cb = {r: cov(ch_b, r) for r in g}
    na = {r: ca[r] @ w for r in g}; nb = {r: cb[r] @ w for r in g}; den = {r: rmask[r] @ w for r in g}
    ka = {r: ca[r].sum() for r in g}; kb = {r: cb[r].sum() for r in g}
    def stat(sample):
        return (sum(na[r] - nb[r] for r in sample) / sum(den[r] for r in sample),
                sum(ka[r] - kb[r] for r in sample))
    d0 = stat(g)
    vals = np.array([stat(rng.choice(g, size=len(g))) for _ in range(n)])
    lo, hi = np.percentile(vals, [2.5, 97.5], axis=0)
    return {"score_diff": d0[0], "score_lo": lo[0], "score_hi": hi[0],
            "covered_diff": d0[1], "covered_lo": lo[1], "covered_hi": hi[1]}

## 7. Tabular baseline (and the non-LLM models, if their scores are present)

In [ ]:
def oof_logistic(make_blocks, C=0.1):
    p = np.zeros(H)
    for tr, te in FOLDS:
        A, B = make_blocks(tr, te)
        sc = StandardScaler().fit(A)
        A, B = np.nan_to_num(sc.transform(A)), np.nan_to_num(sc.transform(B))
        m = LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit(A, Y[tr])
        p[te] = m.predict_proba(B)[:, 1]
    return p

SCORES, SEED_SCORES, FAMILY = {}, {}, {}
SCORES["tabular, logistic"] = oof_logistic(lambda tr, te: (XT[tr], XT[te]))
FAMILY["tabular, logistic"] = ("tabular", None)
print(f"tabular, logistic AUC: {roc_auc_score(Y, SCORES['tabular, logistic']):.3f}")

# The non-LLM notebook's models, if its out-of-fold scores were copied next to the results.
if NONLLM_OOF.exists():
    o = pd.read_csv(NONLLM_OOF).set_index("hunk").loc[hunks["hunk"]]
    if (o["fold"].to_numpy() == FOLD_ID).all() and (o[f"y_{'wider' if LABEL == 'wider' else 'wider_weighted'}"].to_numpy() == Y).all():
        for c in o.columns:
            if c.startswith("score|") and c != "score|tabular, logistic":
                SCORES[f"non-LLM: {c[6:]}"] = o[c].to_numpy()
                FAMILY[f"non-LLM: {c[6:]}"] = ("non-LLM", None)
        assert np.allclose(o["score|tabular, logistic"], SCORES["tabular, logistic"], atol=1e-6), \
            "the tabular baseline differs from the non-LLM notebook's"
        print(f"Added {sum(k.startswith('non-LLM') for k in SCORES)} non-LLM models from {NONLLM_OOF.name}")
    else:
        print(f"{NONLLM_OOF.name} has other folds or labels; not used.")
else:
    print(f"No {NONLLM_OOF.name} in {OUT_DIR}; the non-LLM models are left out of this comparison.")

## 8. LLM helpers

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification

AMP_DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
DTYPE = AMP_DTYPE if DEVICE == "cuda" else torch.float16 if DEVICE == "mps" else torch.float32
FT_DTYPE = AMP_DTYPE if DEVICE == "cuda" else torch.float32
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
_tv = tuple(int(x) for x in transformers.__version__.split(".")[:2])
_DT_NAME = "dtype" if _tv >= (4, 56) else "torch_dtype"
DT_KW, FT_DT_KW = {_DT_NAME: DTYPE}, {_DT_NAME: FT_DTYPE}
print("inference dtype:", DTYPE, "| fine-tuning dtype:", FT_DTYPE)

def size_of(name):
    """'Qwen/Qwen2.5-Coder-7B-Instruct' -> '7B'."""
    return next((p for p in name.replace("/", "-").split("-") if p[:-1].replace(".", "").isdigit() and p.endswith("B")), name)

def slug(name):
    return name.replace("/", "_")

def load_tok(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok

def text_hash(texts):
    return hashlib.sha1("\x00".join(texts).encode()).hexdigest()[:10]

def free():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()

def batches_by_length(lengths, bs):
    order = np.argsort(lengths)
    return [order[i:i + bs] for i in range(0, len(order), bs)]

TIMES = {}

## 9. Zero-shot prompting (no training)

In [ ]:
ZS_QUESTION = (
    "A code reviewer will review the diff hunk below. It can see only the diff and the pull request "
    "description, or it can also be given more code: the function the hunk changes, the whole file, "
    "or the definitions of what the hunk calls. More code costs more. Would more code likely help the "
    "reviewer find real problems in this hunk that it would otherwise miss? Answer with Yes or No only.")

def zero_shot_scores(model_name):
    tok = load_tok(model_name)
    texts = build_texts(tok, MAX_LEN)
    cache = CACHE_DIR / f"zs_{slug(model_name)}_{MAX_LEN}_{text_hash(texts)}.npy"
    if cache.exists():
        print("cached:", cache.name); return np.load(cache)
    model = AutoModelForCausalLM.from_pretrained(model_name, **DT_KW).to(DEVICE).eval()
    prompts = [tok.apply_chat_template(
        [{"role": "system", "content": "You are an expert code reviewer."},
         {"role": "user", "content": ZS_QUESTION + "\n\n" + t}],
        tokenize=False, add_generation_prompt=True) for t in texts]
    yes = [tok.encode(w, add_special_tokens=False)[0] for w in ("Yes", " Yes")]
    no = [tok.encode(w, add_special_tokens=False)[0] for w in ("No", " No")]
    enc = [tok(p, add_special_tokens=False).input_ids for p in prompts]
    out = np.zeros(H)
    with torch.no_grad():
        for idx in batches_by_length([len(e) for e in enc], ZS_BATCH):
            batch = tok.pad({"input_ids": [enc[i] for i in idx]}, return_tensors="pt").to(DEVICE)
            logits = model(**batch).logits
            last = batch["attention_mask"].sum(1) - 1                       # right padding
            lp = torch.log_softmax(logits[torch.arange(len(idx)), last].float(), dim=-1)
            out[idx] = (torch.logsumexp(lp[:, yes], 1) - torch.logsumexp(lp[:, no], 1)).cpu().numpy()
            del logits, batch
    del model; free()
    np.save(cache, out)
    return out

if RUN_ZERO_SHOT:
    for m in ZS_MODELS:
        t0 = time.time()
        s = zero_shot_scores(m)
        name = f"zero-shot {size_of(m)}"
        SCORES[name] = s; FAMILY[name] = ("zero-shot", size_of(m)); TIMES[name] = time.time() - t0
        print(f"{name}: AUC {roc_auc_score(Y, s):.3f}, answers Yes for {(s > 0).mean():.0%} of hunks "
              f"({TIMES[name]:.0f} s)")

## 10. Frozen embeddings + logistic regression

In [ ]:
def embed(model_name):
    tok = load_tok(model_name)
    texts = build_texts(tok, MAX_LEN)
    cache = CACHE_DIR / f"emb_{slug(model_name)}_{MAX_LEN}_{text_hash(texts)}.npy"
    if cache.exists():
        print("cached:", cache.name); return np.load(cache)
    model = AutoModel.from_pretrained(model_name, **DT_KW).to(DEVICE).eval()
    enc = [tok(t, add_special_tokens=False, truncation=True, max_length=MAX_LEN).input_ids for t in texts]
    E = None
    with torch.no_grad():
        for idx in batches_by_length([len(e) for e in enc], EMB_BATCH):
            batch = tok.pad({"input_ids": [enc[i] for i in idx]}, return_tensors="pt").to(DEVICE)
            hs = model(**batch).last_hidden_state.float()
            m = batch["attention_mask"].unsqueeze(-1).float()
            v = ((hs * m).sum(1) / m.sum(1)).cpu().numpy()
            if E is None:
                E = np.zeros((H, v.shape[1]), dtype=np.float32)
            E[idx] = v
    del model; free()
    np.save(cache, E)
    return E

def emb_blocks(E, with_tab):
    def f(tr, te):
        pca = PCA(n_components=min(EMB_PCA, len(tr) - 1), random_state=0).fit(E[tr])
        a, b = pca.transform(E[tr]), pca.transform(E[te])
        return (np.hstack([XT[tr], a]), np.hstack([XT[te], b])) if with_tab else (a, b)
    return f

if RUN_EMBED:
    for m in EMB_MODELS:
        t0 = time.time()
        E = embed(m)
        for with_tab in (False, True):
            name = f"embedding{' + tabular' if with_tab else ''} {size_of(m)}"
            SCORES[name] = oof_logistic(emb_blocks(E, with_tab))
            FAMILY[name] = ("embedding + tabular" if with_tab else "embedding", size_of(m))
            print(f"{name}: AUC {roc_auc_score(Y, SCORES[name]):.3f}")
        TIMES[f"embedding {size_of(m)}"] = time.time() - t0
        print(f"  embeddings {E.shape} ({TIMES[f'embedding {size_of(m)}']:.0f} s)")

## 11. LoRA fine-tuning

The longest step. Each (model, seed, fold) is cached as soon as it finishes.

In [ ]:
def check_peft():
    """Fail early, with the fix, if peft cannot add LoRA on this runtime."""
    import peft
    from peft.import_utils import is_torchao_available
    try:
        is_torchao_available()
    except ImportError as e:
        raise RuntimeError(
            f"peft {peft.__version__} rejects the torchao on this runtime ({e}).\n"
            "Fix: run  !pip install -U 'peft>=0.22'  and  !pip uninstall -y torchao , then "
            "Runtime -> Restart session and Run all. Finished models are cached on Drive.") from None
    print("peft", peft.__version__, "ok")

def finetune_fold(model_name, texts, tok, tr, te, seed):
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import get_linear_schedule_with_warmup
    torch.manual_seed(seed); np.random.seed(seed)
    if SMOKE:
        tr = np.random.default_rng(seed).choice(tr, size=min(FT_SMOKE_TRAIN, len(tr)), replace=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1, **FT_DT_KW)
    model.config.pad_token_id = tok.pad_token_id
    if GRAD_CKPT:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    for p in model.parameters():                     # trainable weights in float32
        if p.requires_grad:
            p.data = p.data.float()
    model.to(DEVICE)

    enc = [tok(texts[i], add_special_tokens=False, truncation=True, max_length=MAX_LEN).input_ids for i in range(H)]
    y = torch.tensor(Y, dtype=torch.float32)
    pos = max(1, int(Y[tr].sum()))
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(tr) - pos) / pos, device=DEVICE))
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=FT_LR, weight_decay=0.0)
    steps_per_epoch = math.ceil(len(tr) / (FT_BATCH * FT_GRAD_ACC))
    sched = get_linear_schedule_with_warmup(opt, int(FT_WARMUP * steps_per_epoch * FT_EPOCHS),
                                            steps_per_epoch * FT_EPOCHS)
    use_amp = DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and AMP_DTYPE == torch.float16)
    rng = np.random.default_rng(seed)

    def run_batch(idx):
        batch = tok.pad({"input_ids": [enc[i] for i in idx]}, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=use_amp):
            return model(**batch).logits.squeeze(-1).float()

    model.train()
    for epoch in range(FT_EPOCHS):
        perm = rng.permutation(tr)                   # shuffle, then batch similar lengths together
        chunks = [perm[i:i + FT_BATCH * 8] for i in range(0, len(perm), FT_BATCH * 8)]
        bl = [c[np.argsort([len(enc[i]) for i in c])][j:j + FT_BATCH]
              for c in chunks for j in range(0, len(c), FT_BATCH)]
        bl = [bl[i] for i in rng.permutation(len(bl))]
        for s, idx in enumerate(bl):
            loss = loss_fn(run_batch(idx), y[torch.as_tensor(idx)].to(DEVICE)) / FT_GRAD_ACC
            scaler.scale(loss).backward()
            if (s + 1) % FT_GRAD_ACC == 0 or s + 1 == len(bl):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()

    model.eval()
    out = np.zeros(len(te))
    with torch.no_grad():
        for idx in batches_by_length([len(enc[i]) for i in te], FT_BATCH * 2):
            out[idx] = run_batch(te[idx]).cpu().numpy()
    del model, opt, scaler; free()
    return out

if RUN_FINETUNE and FT_MODELS:
    check_peft()
    for m in FT_MODELS:
        tok_ft = load_tok(m)
        texts_ft = build_texts(tok_ft, MAX_LEN)
        name = f"LoRA {size_of(m)}"
        SEED_SCORES[name] = []
        t_all = time.time()
        for seed in FT_SEEDS:
            p = np.zeros(H)
            for f_i, (tr, te) in enumerate(FOLDS):
                cache = CACHE_DIR / f"ft_{slug(m)}_{FT_TAG}_{FOLD_HASH}_{text_hash(texts_ft)}_s{seed}_f{f_i}.npy"
                if cache.exists():
                    p[te] = np.load(cache); continue
                t0 = time.time()
                p[te] = finetune_fold(m, texts_ft, tok_ft, tr, te, seed)
                np.save(cache, p[te])
                print(f"{name} seed {seed} fold {f_i + 1}/{len(FOLDS)}: {time.time() - t0:.0f} s")
            SEED_SCORES[name].append(p)
            print(f"{name} seed {seed}: AUC {roc_auc_score(Y, p):.3f}")
        # Seeds are combined by averaging ranks (logit scales differ between runs).
        SCORES[name] = np.mean([pd.Series(p).rank().to_numpy() for p in SEED_SCORES[name]], axis=0)
        FAMILY[name] = ("LoRA", size_of(m)); TIMES[name] = time.time() - t_all
        print(f"{name}: AUC {roc_auc_score(Y, SCORES[name]):.3f} ({TIMES[name]:.0f} s)")

## 12. Can a model tell which hunks need context? (AUC)

In [ ]:
# AUC of every method, with a repository-clustered 95% interval.
auc_rows = []
for name, s in SCORES.items():
    lo, hi = cluster_boot_auc(Y, s, REPOS)
    seeds = [roc_auc_score(Y, p) for p in SEED_SCORES.get(name, [])]
    auc_rows.append({"method": name, "auc": roc_auc_score(Y, s), "ci_low": lo, "ci_high": hi,
                     "auc_seed_mean": np.mean(seeds) if seeds else np.nan,
                     "auc_seed_sd": np.std(seeds) if len(seeds) > 1 else np.nan})
for up in UPGRADE_LEVELS:
    lo, hi = cluster_boot_auc(Y, CHEAP[up], REPOS)
    auc_rows.append({"method": f"cheapest first to {up} (no learning)", "auc": roc_auc_score(Y, CHEAP[up]),
                     "ci_low": lo, "ci_high": hi})
AUC = pd.DataFrame(auc_rows).sort_values("auc", ascending=False).reset_index(drop=True)
print(AUC.round(3).to_string(index=False))

## 13. At a token budget

In [ ]:
# Every method at every spend, for every upgrade level.
named = {**SCORES}
rows_b = []
for up in UPGRADE_LEVELS:
    pols = {**named, "cheapest first (no learning)": CHEAP[up], "hindsight (upper bound)": ORACLE[up]}
    for name, s in pols.items():
        for _, r in frontier(s, up).iterrows():
            rows_b.append({"method": name, **r.to_dict(), **RAND[(up, r["spend"])]})
BUDGET_TABLE = pd.DataFrame(rows_b)
sc = f"score_{TARGET_WEIGHTING}"
BUDGET_TABLE["beats_random_p95"] = BUDGET_TABLE[sc] > BUDGET_TABLE["score_p95"]
for up in UPGRADE_LEVELS:
    BUDGET_TABLE.loc[BUDGET_TABLE.upgrade_to == up, "always_up_score"] = FIXED.loc[LIDX[up], sc]
    BUDGET_TABLE.loc[BUDGET_TABLE.upgrade_to == up, "always_up_covered"] = FIXED.loc[LIDX[up], "covered"]
BUDGET_TABLE["matches_always_up"] = BUDGET_TABLE[sc] >= BUDGET_TABLE["always_up_score"] - 1e-9

for up in UPGRADE_LEVELS:
    t = BUDGET_TABLE[(BUDGET_TABLE.upgrade_to == up) & (BUDGET_TABLE.spend == TARGET_SPEND)]
    r0 = RAND[(up, TARGET_SPEND)]
    print(f"\nUpgrade to {up}, spending {TARGET_SPEND:.0%} of its extra tokens "
          f"(always {BASE_LEVEL}: {FIXED.loc[b_i, 'covered']} refs, score {FIXED.loc[b_i, sc]:.3f}; "
          f"always {up}: {FIXED.loc[LIDX[up], 'covered']} refs, score {FIXED.loc[LIDX[up], sc]:.3f}; "
          f"random: score mean {r0['score_mean']:.3f}, p95 {r0['score_p95']:.3f})")
    print(t.sort_values(sc, ascending=False)[["method", "covered", sc, "recall_human", "recall_ai",
          "token_share", "beats_random_p95", "matches_always_up"]].round(3).to_string(index=False))

## 14. Small against large models

In [ ]:
# Small against large, per family: AUC and the score at the target spend for the main upgrade level.
sc = f"score_{TARGET_WEIGHTING}"
t = BUDGET_TABLE[(BUDGET_TABLE.upgrade_to == MAIN_UP) & (BUDGET_TABLE.spend == TARGET_SPEND)].set_index("method")
auc = AUC.set_index("method")
size_rows = []
for name, (fam, size) in FAMILY.items():
    size_rows.append({"family": fam, "size": size or "", "method": name, "auc": auc.loc[name, "auc"],
                      "ci_low": auc.loc[name, "ci_low"], "ci_high": auc.loc[name, "ci_high"],
                      "auc_seed_sd": auc.loc[name, "auc_seed_sd"],
                      f"covered@{TARGET_SPEND:.0%}": t.loc[name, "covered"], f"score@{TARGET_SPEND:.0%}": t.loc[name, sc],
                      "beats_random_p95": t.loc[name, "beats_random_p95"], "seconds": TIMES.get(name, np.nan)})
SIZE_TABLE = pd.DataFrame(size_rows)
order_f = {"tabular": 0, "non-LLM": 1, "zero-shot": 2, "embedding": 3, "embedding + tabular": 4, "LoRA": 5}
SIZE_TABLE["_s"] = SIZE_TABLE["size"].map(lambda s: float(s[:-1]) if s and s[:-1].replace(".", "").isdigit() else 0)
SIZE_TABLE = SIZE_TABLE.sort_values(["family", "_s"], key=lambda c: c.map(order_f) if c.name == "family" else c).drop(columns="_s")
print(f"Upgrade to {MAIN_UP} at {TARGET_SPEND:.0%} of its extra tokens "
      f"(always {BASE_LEVEL}: score {FIXED.loc[b_i, sc]:.3f}; always {MAIN_UP}: {FIXED.loc[LIDX[MAIN_UP], sc]:.3f}; "
      f"random p95: {RAND[(MAIN_UP, TARGET_SPEND)]['score_p95']:.3f})")
print(SIZE_TABLE.round(3).to_string(index=False))

## 15. Paired bootstrap and reference groups

In [ ]:
# Paired, repository-clustered bootstrap at the target spend, for the main upgrade level.
POL = {name: upgrade_choice(order_of(s), MAIN_UP, TARGET_SPEND) for name, s in SCORES.items()}
others = {f"always {MAIN_UP}": fixed(MAIN_UP), f"always {BASE_LEVEL}": fixed(BASE_LEVEL),
          "cheapest first": upgrade_choice(order_of(CHEAP[MAIN_UP]), MAIN_UP, TARGET_SPEND)}
if REFERENCE_METHOD in POL:
    others[REFERENCE_METHOD] = POL[REFERENCE_METHOD]
cmp_rows = []
for name in SCORES:
    for oname, och in others.items():
        if oname == name:
            continue
        cmp_rows.append({"method": name, "versus": oname, **boot_diff(POL[name], och)})
COMPARE = pd.DataFrame(cmp_rows)
print(f"Upgrade to {MAIN_UP} at {TARGET_SPEND:.0%} spend; differences with 95% intervals")
print(COMPARE.round(3).to_string(index=False))

# Human and AI references, tiers, context labels and categories at the same point.
def group_table(policies):
    out = []
    for name, ch in policies.items():
        covered = COV[np.arange(H), ch, :].any(axis=0)
        row = {"policy": name}
        for col in ("human", "tier", "context", "category"):
            for val, idx in refs_df.groupby(refs_df[col].astype(str)).groups.items():
                row[f"{col}={val}"] = f"{int(covered[idx].sum())}/{len(idx)}"
        out.append(row)
    return pd.DataFrame(out).set_index("policy").T
GROUPS = group_table({**{f"always {lv}": fixed(lv) for lv in LEVELS}, **POL, "cheapest first": others["cheapest first"]})
print(GROUPS.loc[[i for i in GROUPS.index if i.startswith(("human", "tier"))]].to_string())

## 16. Quality-cost curves

In [ ]:
sc = f"score_{TARGET_WEIGHTING}"
show = [m for m in SCORES if m in PLOT_METHODS] or list(SCORES)[:6]
fig, axes = plt.subplots(1, len(UPGRADE_LEVELS), figsize=(6 * len(UPGRADE_LEVELS), 4.6), sharey=True)
for ax, up in zip(np.atleast_1d(axes), UPGRADE_LEVELS):
    t = BUDGET_TABLE[BUDGET_TABLE.upgrade_to == up]
    rr = t[t.method == show[0]]
    ax.fill_between(rr["spend"], [RAND[(up, f)]["score_mean"] for f in rr["spend"]],
                    [RAND[(up, f)]["score_p95"] for f in rr["spend"]], color="0.85", label="random (mean to p95)")
    for m in show + ["cheapest first (no learning)", "hindsight (upper bound)"]:
        q = t[t.method == m]
        style = "--" if m.startswith(("cheapest", "hindsight")) else "-"
        ax.plot(q["spend"], q[sc], style, marker="o", ms=3, label=m)
    ax.axhline(FIXED.loc[LIDX[up], sc], color="k", lw=0.8, ls=":", label=f"always {up}")
    ax.axhline(FIXED.loc[b_i, sc], color="0.5", lw=0.8, ls=":", label=f"always {BASE_LEVEL}")
    ax.set_title(f"upgrade {BASE_LEVEL} -> {up}")
    ax.set_xlabel("share of the extra tokens spent"); ax.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel(f"severity score ({TARGET_WEIGHTING})")
np.atleast_1d(axes)[-1].legend(fontsize=7, loc="lower right")
plt.tight_layout()
fig.savefig(OUT_DIR / FIG_NAME, dpi=150)
plt.show()

## 17. Save

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

pre = "smoke_" if SMOKE else ""
AUC.to_csv(OUT_DIR / f"{pre}llm_auc.csv", index=False)
SIZE_TABLE.to_csv(OUT_DIR / f"{pre}llm_size_comparison.csv", index=False)
BUDGET_TABLE.to_csv(OUT_DIR / f"{pre}llm_budget_table.csv", index=False)
COMPARE.to_csv(OUT_DIR / f"{pre}llm_bootstrap.csv", index=False)
GROUPS.to_csv(OUT_DIR / f"{pre}llm_groups.csv")
oof_df = pd.DataFrame({"hunk": hunks["hunk"], "pr": hunks["pr"], "repo": REPOS, "fold": FOLD_ID,
                       "y_wider": Y_COUNT, "y_wider_weighted": Y_WEIGHTED})
for name, s in SCORES.items():
    oof_df[f"score|{name}"] = s
for name, runs in SEED_SCORES.items():
    for seed, p in zip(FT_SEEDS, runs):
        oof_df[f"score|{name}|seed{seed}"] = p
oof_df.to_csv(OUT_DIR / f"{pre}llm_oof_scores.csv", index=False)
manifest = {
    "written_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    "inputs": {k: str(v) for k, v in FILES.items()},
    "sha256": {k: sha256(v) for k, v in FILES.items()},
    "hunks": H, "references": R, "label": LABEL, "positives": int(Y.sum()),
    "cv": CV, "folds": len(FOLDS), "fold_hash": FOLD_HASH, "base_level": BASE_LEVEL,
    "upgrade_levels": UPGRADE_LEVELS, "main_upgrade": MAIN_UP, "spend": SPEND, "target_spend": TARGET_SPEND,
    "target_weighting": TARGET_WEIGHTING, "n_random": N_RANDOM, "n_boot": N_BOOT,
    "zs_models": ZS_MODELS if RUN_ZERO_SHOT else [], "emb_models": EMB_MODELS if RUN_EMBED else [],
    "ft_models": FT_MODELS if RUN_FINETUNE else [], "max_len": MAX_LEN, "diff_share": DIFF_SHARE,
    "use_meta": USE_META, "emb_pca": EMB_PCA,
    "finetune": {"seeds": FT_SEEDS, "epochs": FT_EPOCHS, "lr": FT_LR, "batch": FT_BATCH, "grad_acc": FT_GRAD_ACC,
                 "lora": [LORA_R, LORA_ALPHA, LORA_DROPOUT], "tag": FT_TAG, "grad_ckpt": GRAD_CKPT},
    "smoke": SMOKE, "seconds": TIMES,
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                 "sklearn": sklearn.__version__, "torch": torch.__version__,
                 "transformers": transformers.__version__, "gpu": GPU_NAME, "dtype": str(DTYPE)},
    "note": "AACR-Bench 50% (93 PRs, 928 hunks, four levels). One reviewer run. No locked test set.",
}
(OUT_DIR / f"{pre}llm_manifest.json").write_text(json.dumps(manifest, indent=1))
print("Saved to", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print("  ", p.name)